<div style="background: linear-gradient(90deg, #075e5e 0%, #0f7774 100%); color: white; padding: 28px 36px; border-radius: 18px; border-left: 12px solid #063f40; box-shadow: 0 8px 22px rgba(0,0,0,0.18);">
  <h1 style="margin: 0; font-size: 34px; font-weight: 800;">Hệ Thống Gợi Ý Phim </h1>
</div>


## Mục tiêu của hệ thống
Hệ thống có thể gợi ý phim theo hai cách:

1. **Gợi ý theo tên phim**  
   Người dùng nhập tên một bộ phim, hệ thống tìm các phim có nội dung tương tự.

2. **Gợi ý theo truy vấn tự do**  
   Người dùng nhập các thông tin như thể loại, mô tả nội dung, đạo diễn, diễn viên hoặc quốc gia sản xuất, hệ thống trả về các phim phù hợp nhất.

## Cấu trúc thuật toán

### 1. Trích xuất đặc trưng (Feature Extraction)
Mô hình sử dụng các trường dữ liệu sau để tính toán độ tương đồng Cosine (Cosine Similarity):
- `Genre` (Thể loại): Mã hóa dạng Multi-hot.
- `Director` (Đạo diễn): Mã hóa dạng Multi-hot (chỉ giữ lại đạo diễn có từ 3 phim trở lên để giảm nhiễu).
- `Stars` (Diễn viên): Mã hóa dạng Multi-hot (chỉ giữ lại top 300 diễn viên phổ biến nhất).
- `Countries_of_origin` (Quốc gia): Mã hóa dạng Multi-hot.
- `Overview` (Tóm tắt nội dung): Trích xuất vector ngữ nghĩa bằng mô hình SBERT.

### 2. Loại bỏ các yếu tố nhiễu trong tính toán khoảng cách
Để đảm bảo tính khách quan của độ tương đồng nội dung, các biến số dạng số (Numeric) không được đưa vào vector nội dung:
- Không sử dụng `Runtime_min` (Thời lượng).
- Không sử dụng `Weighted_Rating` (Điểm đánh giá) và `Release_Year` (Năm phát hành) trong ma trận đặc trưng `X_all`.

### 3. Chiến lược gợi ý 2 giai đoạn (Retrieval & Reranking)
- **Giai đoạn 1 (Retrieval):** Sử dụng thuật toán K-Nearest Neighbors (KNN) để truy xuất danh sách **Top 50** bộ phim có nội dung tương đồng nhất với truy vấn.
- **Giai đoạn 2 (Reranking):** Xếp hạng lại danh sách 50 bộ phim này dựa trên công thức kết hợp điểm nội dung và chất lượng phim:
  `Điểm tổng hợp (Final Score) = 0.8 * Cosine_Similarity + 0.2 * MinMax_Scale(Weighted_Rating)`
- **Kết quả:** Trích xuất **Top 10** bộ phim có điểm tổng hợp cao nhất để gợi ý cho người dùng.

Trong đó:

- `CosineSimilarity`: mức độ giống nhau về nội dung giữa phim truy vấn và phim ứng viên.
- `Weighted_Rating`: điểm đánh giá của phim sau khi được chuẩn hóa về khoảng `[0, 1]`.
- `alpha_similarity`: trọng số ưu tiên độ tương đồng nội dung.
- `beta_weighted_rating`: trọng số ưu tiên điểm đánh giá.

## Quy trình gợi ý phim

1. Tiền xử lý dữ liệu và chuẩn hóa các cột văn bản.
2. Chuyển các cột đa giá trị thành danh sách.
3. Mã hóa các cột có nhiều giá trị bằng phương pháp MultiLabelBinarizer
4. Mã hóa mô tả phim bằng SBERT.
5. Ghép các nhóm đặc trưng thành vector nội dung tổng hợp.
6. Dùng KNN để lấy danh sách phim ứng viên dựa trên độ tương đồng cosine.
7. Xếp hạng lại các phim ứng viên bằng cách kết hợp:
   - `CosineSimilarity`: mức độ giống nhau về nội dung.
   - `Weighted_Rating`: điểm đánh giá đã được chuẩn hóa.

## 1. Cài đặt thư viện

In [ ]:
#pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Khai báo thư viện

In [ ]:
import re
import ast
import numpy as np
import pandas as pd

from collections import Counter
from scipy.sparse import csr_matrix, hstack

from sklearn.preprocessing import normalize, MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer


## 3. Đọc và kiểm tra dữ liệu 

In [ ]:
# =========================
# Đọc và kiểm tra dữ liệu đầu vào 
# =========================

DATA_PATH = "D:\py\DeAn1\PhanTich\HeThongDeXuat\movies_recommender_dataset.csv"
df = pd.read_csv(DATA_PATH)

required_cols = [
    "Movie_Title",
    "Genre",
    "Director",
    "Stars",
    "Overview",
    "Release_Year",
    "Weighted_Rating",
    "Countries_of_origin",
]

# Kiểm tra xem dữ liệu có bị thiếu cột nào không
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Thiếu cột trong dữ liệu: {missing_cols}")

print(f"Số dòng dữ liệu: {len(df)}")
print(f"Số cột: {len(df.columns)}")
df.head(3)


Số dòng dữ liệu: 61342
Số cột: 10


,ID,Movie_Title,Genre,Director,Stars,Overview,Weighted_Rating,Runtime_min,Release_Year,Countries_of_origin
0,MV1,The Praetorian,Thriller,Nelson Ricardo,Nelson Ricardo | Bello Patricia | Nicholas Apo...,David Donovan was a Bodyguard for hire who wil...,6.293896,94.0,2020,United States
1,MV2,365 Days,Drama|Romance,Barbara Bialowas | Tomasz Mandes,Anna-Maria Sieklucka | Michele Morrone | Broni...,Massimo is a member of the Sicilian Mafia fami...,3.327324,114.0,2020,Poland
2,MV3,Promising Young Woman,Crime|Drama|Mystery,Emerald Fennell,Carey Mulligan | Bo Burnham | Alison Brie,An unexpected encounter gives a wickedly smart...,7.495181,113.0,2020,United Kingdom|United States


## 4. Định nghĩa các hàm tiền xử lý

In [ ]:
# =========================
# Các hàm tiền xử lý
# =========================

def normalize_text(text: str) -> str:
    """Chuẩn hóa văn bản: chuyển chữ thường, loại bỏ ký tự đặc biệt."""    
    text = str(text).lower().strip() 
    text = re.sub(r"[^a-z0-9\s]", " ", text) 
    text = re.sub(r"\s+", " ", text).strip()  #Chuẩn hóa khoảng trắng: Gom nhiều khoảng trắng liên tiếp thành một khoảng trắng
    return text
   
   
# Chuẩn hóa tên riêng như tên đạo diễn, diễn viên hoặc quốc gia: tránh bị tách nhiều phần rời rạc khi mã hóa 
    # Sau khi chuẩn hóa, các từ trong tên được nối bằng dấu gạch dưới.
    # Ví dụ: "Christopher Nolan" -> "christopher_nolan".
def normalize_name_token(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[-/]", " ", name)
    name = re.sub(r"[^a-z0-9\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name.replace(" ", "_")


def try_parse_list_string(text):
    """
    Chuyển một chuỗi dạng danh sách thành list Python.
    Một số bộ dữ liệu lưu dữ liệu dạng chuỗi như:
        "['Action', 'Drama']"
    Nếu chuỗi đúng định dạng list, hàm sẽ trả về list các phần tử.
    Nếu không đúng định dạng, hàm trả về None để xử lý bằng cách tách chuỗi thông thường.
    """
    if pd.isna(text):
        return None
    text = str(text).strip()
    if not text:
        return None

    if text.startswith("[") and text.endswith("]"):
        try:
            value = ast.literal_eval(text)
            if isinstance(value, list):
                return [str(x).strip() for x in value if str(x).strip()]
        except Exception:
            pass
    return None


def split_multivalue(text: str):
    """
    Tách một ô dữ liệu chứa một hoặc nhiều giá trị (ngăn cách bởi dấu | hoặc ,) thành danh sách các giá trị đơn lẻ
    -> ['Action', 'Drama', 'Crime']
    Các giá trị rỗng hoặc không rõ nghĩa như "nan", "unknown", "none" sẽ bị loại bỏ.
    """
    if pd.isna(text):
        return []

    parsed = try_parse_list_string(text)
    if parsed is not None:
        raw_items = parsed
    else:
        text = str(text).strip()
        if text == "":
            return []

        if "|" in text:
            raw_items = text.split("|")
        elif "," in text:
            raw_items = text.split(",")
        else:
            raw_items = [text]

    cleaned = []
    for item in raw_items:
        item = str(item).strip()
        if item.lower() in {"", "nan", "unknown", "none"}:
            continue
        cleaned.append(item)
    return cleaned


def preprocess_overview(text: str) -> str:
    """
    Làm sạch cột mô tả nội dung phim.
    Nếu mô tả bị thiếu hoặc không có ý nghĩa, hàm trả về chuỗi rỗng.
    Ngược lại, mô tả được chuẩn hóa bằng hàm normalize_text().
    """
    if pd.isna(text):
        return ""
    text = str(text).strip()
    if text.lower() in {"", "nan", "unknown", "no description"}:
        return ""
    return normalize_text(text)


def preprocess_tag_list(values, mode="name"):
    """
    Làm sạch danh sách nhãn trước khi mã hóa.

    Hàm sẽ chuẩn hóa từng giá trị trong danh sách:
    - Nếu là tên riêng như đạo diễn, diễn viên, quốc gia thì chuẩn hóa theo hàm tên.
    - Nếu là dữ liệu văn bản như thể loại thì chuẩn hóa theo hàm văn bản thường

    Sau đó, hàm loại bỏ các giá trị rỗng và các phần tử bị trùng lặp.
    Kết quả trả về là danh sách nhãn đã được làm sạch.
    """
    cleaned = []
    for v in values:
        if mode == "name":
            token = normalize_name_token(v)
        else:
            token = normalize_text(v)
        if token:
            cleaned.append(token)
    return list(dict.fromkeys(cleaned))


def minmax_scale_safe(series):
    """
    Chuẩn hóa dữ liệu số về khoảng [0, 1] theo Min-Max Scaling.
    Hàm được dùng để chuẩn hóa Weighted_Rating trước khi tính FinalScore.
    """
    s = pd.Series(series).astype(float)
    min_v = s.min()
    max_v = s.max()

    if pd.isna(min_v) or pd.isna(max_v):
        return pd.Series(np.zeros(len(s)), index=s.index)
    if max_v == min_v:
        return pd.Series(np.ones(len(s)), index=s.index)

    return (s - min_v) / (max_v - min_v)


## Tiền xử lý dữ liệu thực tế 

In [ ]:
# =========================
# Áp dụng hàm đã khai báo, chuyển các cột đa giá trị về dạng danh sách
# =========================

# Dùng hàm split_multivalue để tách các ô có nhiều giá trị thành danh sách.
df["Genre_list_raw"] = df["Genre"].apply(split_multivalue)
df["Director_list_raw"] = df["Director"].apply(split_multivalue)
df["Stars_list_raw"] = df["Stars"].apply(split_multivalue)
df["Countries_of_origin_list_raw"] = df["Countries_of_origin"].apply(split_multivalue)


# Tiếp tục làm sạch danh sách vừa tách: chuẩn hóa chữ viết, bỏ khoảng trắng thừa, bỏ giá trị rỗng, trùng lặp 
df["Genre_list"] = df["Genre_list_raw"].apply(lambda x: preprocess_tag_list(x, mode="text"))
df["Director_list"] = df["Director_list_raw"].apply(lambda x: preprocess_tag_list(x, mode="name"))
df["Stars_list"] = df["Stars_list_raw"].apply(lambda x: preprocess_tag_list(x, mode="name"))
df["Countries_of_origin_list"] = df["Countries_of_origin_list_raw"].apply(lambda x: preprocess_tag_list(x, mode="name"))

# Làm sạch nội dung tóm tắt và tên phim
df["Overview_clean"] = df["Overview"].apply(preprocess_overview)
df["Movie_Title_norm"] = df["Movie_Title"].fillna("").astype(str).str.strip().str.lower()

# Xem thử kết quả 
df[[
    "Movie_Title",
    "Genre_list",
    "Director_list",
    "Stars_list",
    "Countries_of_origin_list",
    "Overview_clean"
]].head(5)


,Movie_Title,Genre_list,Director_list,Stars_list,Countries_of_origin_list,Overview_clean
0,The Praetorian,[thriller],[nelson_ricardo],"[nelson_ricardo, bello_patricia, nicholas_apos...",[united_states],david donovan was a bodyguard for hire who wil...
1,365 Days,"[drama, romance]","[barbara_bialowas, tomasz_mandes]","[anna_maria_sieklucka, michele_morrone, bronis...",[poland],massimo is a member of the sicilian mafia fami...
2,Promising Young Woman,"[crime, drama, mystery]",[emerald_fennell],"[carey_mulligan, bo_burnham, alison_brie]","[united_kingdom, united_states]",an unexpected encounter gives a wickedly smart...
3,The Father,"[drama, mystery]",[florian_zeller],"[anthony_hopkins, olivia_colman, mark_gatiss]","[united_kingdom, france, united_states]",a man refuses all assistance from his daughter...
4,The Invisible Man,"[drama, horror, mystery]",[leigh_whannell],"[elisabeth_moss, oliver_jackson_cohen, harriet...","[australia, united_states]",when cecilia s abusive ex takes his own life a...


## 6. Giảm chiều dữ liệu nhiễu

In [ ]:
# =========================
# Lọc Director và Stars để giảm nhiễu
# =========================

# Đếm tần suất và chỉ giữ lại các đạo diễn xuất hiện từ 3 lần trở lên (Có từ 3 phim trở lên)
director_counter = Counter()
for items in df["Director_list"]:
    director_counter.update(items)

valid_directors = {name for name, cnt in director_counter.items() if cnt >= 3}

# Đếm tần suất và giữ lại 300 diễn viên xuất hiện nhiều nhất.
star_counter = Counter()
for items in df["Stars_list"]:
    star_counter.update(items)

top_300_stars = {name for name, _ in star_counter.most_common(300)}

# Áp dụng bộ lọc vào tập dữ liệu 
df["Director_list_f"] = df["Director_list"].apply(
    lambda items: [x for x in items if x in valid_directors]
)
df["Stars_list_f"] = df["Stars_list"].apply(
    lambda items: [x for x in items if x in top_300_stars]
)

print("Số director giữ lại:", len(valid_directors))
print("Số star giữ lại:", len(top_300_stars))


Số director giữ lại: 3577
Số star giữ lại: 300


## 7. Mã hóa đặc trưng phân loại

In [ ]:
# =========================
# MÃ HÓA MULTI-HOT CHO CÁC THUỘC TÍNH PHÂN LOẠI (CATEGORICAL)
# =========================

mlb_genre = MultiLabelBinarizer()
X_genre = csr_matrix(mlb_genre.fit_transform(df["Genre_list"]))

mlb_director = MultiLabelBinarizer()
X_director = csr_matrix(mlb_director.fit_transform(df["Director_list_f"]))

mlb_stars = MultiLabelBinarizer()
X_stars = csr_matrix(mlb_stars.fit_transform(df["Stars_list_f"]))

mlb_origin = MultiLabelBinarizer()
X_origin = csr_matrix(mlb_origin.fit_transform(df["Countries_of_origin_list"]))

print("X_genre:", X_genre.shape)
print("X_director:", X_director.shape)
print("X_stars:", X_stars.shape)
print("X_origin:", X_origin.shape)


X_genre: (61342, 26)
X_director: (61342, 3577)
X_stars: (61342, 300)
X_origin: (61342, 196)


## 8. Mã hóa phần mô tả phim bằng SBERT

In [ ]:
# =========================
# Mã hóa phần mô tả phim bằng SBERT
# =========================

SBERT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

overview_texts = df["Overview_clean"].fillna("").astype(str).tolist()

overview_embeddings = sbert_model.encode(
    overview_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_overview = csr_matrix(overview_embeddings)
print("X_overview:", X_overview.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/959 [00:00<?, ?it/s]

X_overview: (61342, 384)


## 9. Gộp ma trận đặc trưng

In [ ]:
# =========================
# Ghép các đặc trưng (đã được vector hóa) theo trọng số để tạo vector nội dung tổng hợp cho mỗi phim
# Lưu ý: Không đưa các biến dạng số (numeric) vào vector nội dung
# =========================

# Chuẩn hóa độ dài (L2 Norm) cho từng khối đặc trưng
X_genre_n = normalize(X_genre, norm="l2", copy=True)
X_director_n = normalize(X_director, norm="l2", copy=True)
X_stars_n = normalize(X_stars, norm="l2", copy=True)
X_origin_n = normalize(X_origin, norm="l2", copy=True)
X_overview_n = normalize(X_overview, norm="l2", copy=True)

# Gán trọng số mức độ quan trọng cho từng trường dữ liệu
WEIGHTS = {
    "genre": 0.4,
    "overview": 0.4,
    "director": 0.05,
    "stars": 0.05,
    "countries_of_origin": 0.1,
}

# Nối các ma trận thành một ma trận tổng (X_all)
X_all = hstack([
    X_genre_n * WEIGHTS["genre"],
    X_overview_n * WEIGHTS["overview"],
    X_director_n * WEIGHTS["director"],
    X_stars_n * WEIGHTS["stars"],
    X_origin_n * WEIGHTS["countries_of_origin"],
], format="csr")

# Chuẩn hóa L2 lần cuối cho ma trận tổng
X_all = normalize(X_all, norm="l2", copy=False)
print("Kích thước ma trận tổng hợp (X_all):", X_all.shape)

X_all: (61342, 4483)


## 10. Khởi tạo mô hình KNN

In [ ]:
# =========================
# 8) Huấn luyện mô hình KNN với chỉ số là cosine distance
# Dùng để truy xuất (Retrieve) các phim có nội dung tương đồng
# =========================

# Số lượng ứng viên cần lấy ra trong giai đoạn 1 (Giai đoạn Retrieval)
# Cộng thêm 1 để loại trừ chính bộ phim đang được dùng làm truy vấn ra khỏi kết quả 
CANDIDATE_POOL_DEFAULT = 50

nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=CANDIDATE_POOL_DEFAULT + 1
)
nn_model.fit(X_all)

print("Đã fit NearestNeighbors xong.")


Đã fit NearestNeighbors xong.


## 11. Các hàm hỗ trợ tìm kiếm 

In [ ]:
# =========================
# Các hàm hỗ trợ tìm kiếm và xử lý truy vấn
# =========================
# Nhóm hàm này giúp hệ thống:
# - Tìm phim theo tên người dùng nhập.
# - Xử lý trường hợp tên phim nhập chưa chính xác hoàn toàn.
# - Chuyển truy vấn tự do thành dữ liệu phù hợp với MultiLabelBinarizer.


SHOW_COLS = [
    "ID",
    "Movie_Title",
    "Release_Year",
    "Weighted_Rating",
    "Genre",
    "Director",
    "Stars",
    "Countries_of_origin",
]
SHOW_COLS = [c for c in SHOW_COLS if c in df.columns]


def resolve_movie_query(movie_title, release_year=None, verbose=True): #dùng để xác định một phim cụ thể được chọn làm phim truy vấn chính.
    """Xác định chính xác bộ phim mà người dùng muốn tìm (xử lý trường hợp trùng tên)."""
    query = str(movie_title).strip().lower()
    candidates = df[df["Movie_Title_norm"] == query].copy()

    # Lọc theo năm phát hành (nếu người dùng có nhập vào) 
    if release_year is not None:
        candidates_year = candidates[candidates["Release_Year"] == release_year].copy()
        if not candidates_year.empty:
            candidates = candidates_year
            
    # Nếu không tìm thấy tên chính xác, chuyển sang tìm gần đúng
    if candidates.empty:
        candidates = df[df["Movie_Title_norm"].str.contains(query, regex=False, na=False)].copy()
        if release_year is not None and not candidates.empty:
            candidates_year = candidates[candidates["Release_Year"] == release_year].copy()
            if not candidates_year.empty:
                candidates = candidates_year

    if candidates.empty:
        if verbose:
            print("Không tìm thấy phim phù hợp với truy vấn.")
        return None, None
    
    # Ưu tiên phim có đánh giá cao nhất và mới nhất nếu có nhiều kết quả
    candidates = candidates.sort_values(
        by=["Weighted_Rating", "Release_Year"],
        ascending=[False, False]
    )

    chosen_idx = candidates.index[0]

    if verbose:
        print("Danh sách ứng viên truy vấn:")
        display(candidates[["Movie_Title", "Release_Year", "Weighted_Rating"]].head(10))
        print(f"Phim được chọn làm truy vấn là index: {chosen_idx}")

    return chosen_idx, candidates


def transform_multilabel_query(raw_text, mlb, mode="name", allowed_set=None):
    """
    Xử lý dữ liệu người dùng nhập vào và chuyển thành vector nhị phân
    giống với dữ liệu đã được mã hóa bằng MultiLabelBinarizer.
    """
    # Tách chuỗi nhập vào thành danh sách các giá trị riêng lẻ.
    items = split_multivalue(raw_text)

    # Làm sạch và chuẩn hóa các giá trị trong danh sách.
    items = preprocess_tag_list(items, mode=mode)

    # Nếu có danh sách giá trị được phép, chỉ giữ lại các giá trị hợp lệ.
    if allowed_set is not None:
        items = [x for x in items if x in allowed_set]

    # Mã hóa danh sách bằng MultiLabelBinarizer đã được huấn luyện trước đó.
    row = mlb.transform([items])

    return csr_matrix(row), items


### 12. Xây dựng truy vấn linh hoạt: Tạo vector cho truy vấn người dùng khi không nhập trực tiếp tên phim mà chỉ nhập những nội dung khác 

In [ ]:
# =========================
# 10) Xây dựng vector cho truy vấn tự do
# Lưu ý: truy vấn tự do cũng không sử dụng đặc trưng số
# =========================


"""
Tạo vector đặc trưng cho truy vấn tự do của người dùng.

Người dùng có thể nhập một hoặc nhiều thông tin:
- genre: thể loại phim mong muốn.
- overview: mô tả ngắn về nội dung phim.
- director: đạo diễn.
- stars: diễn viên.
- countries_of_origin: quốc gia sản xuất.

Nguyên tắc xử lý:
1. Mỗi nhóm thông tin được mã hóa giống cách đã mã hóa dữ liệu phim.
2. Chỉ những nhóm có dữ liệu mới được tính trọng số.
3. Trọng số của các nhóm đang hoạt động được chuẩn hóa lại để tổng bằng 1.
4. Các nhóm không được nhập sẽ được thay bằng vector 0 để giữ đúng số chiều.
"""
    
def build_query_vector(
    genre="",
    overview="",
    director="",
    stars="",
    countries_of_origin="",
):
    # mã hóa từng nhóm thông tin của do người dùng nhập vào  
    Xq_genre, genre_items = transform_multilabel_query(
        genre, mlb_genre, mode="text"
    )
    Xq_director, director_items = transform_multilabel_query(
        director, mlb_director, mode="name", allowed_set=set(mlb_director.classes_)
    )
    Xq_stars, stars_items = transform_multilabel_query(
        stars, mlb_stars, mode="name", allowed_set=set(mlb_stars.classes_)
    )
    Xq_origin, origin_items = transform_multilabel_query(
        countries_of_origin, mlb_origin, mode="name"
    )

    # Mã hóa mô tả bằng cùng mô hình SBERT đã dùng cho dữ liệu phim. 
    overview_clean = preprocess_overview(overview)
    overview_emb = sbert_model.encode(
        [overview_clean],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    Xq_overview = csr_matrix(overview_emb)

  # Đánh giá lại trọng số cho vector truy vấn dựa trên các trường mà người dùng đã nhập
    active = {}
    if len(genre_items) > 0:
        active["genre"] = WEIGHTS["genre"]
    if overview_clean:
        active["overview"] = WEIGHTS["overview"]
    if len(director_items) > 0:
        active["director"] = WEIGHTS["director"]
    if len(stars_items) > 0:
        active["stars"] = WEIGHTS["stars"]
    if len(origin_items) > 0:
        active["countries_of_origin"] = WEIGHTS["countries_of_origin"]

    if not active:
        raise ValueError("Bạn phải nhập ít nhất một thông tin để gợi ý phim.")

    # Chuẩn hóa lại trọng số để tổng trọng số của các nhóm được nhập bằng 1.
    weight_sum = sum(active.values())
    active = {k: v / weight_sum for k, v in active.items()}

    # Ghép các khối vector theo đúng thứ tự đã dùng khi xây dựng X_all.
    blocks = []
    if "genre" in active:
        blocks.append(normalize(Xq_genre, norm="l2") * active["genre"])
    else:
        blocks.append(csr_matrix((1, X_genre.shape[1])))

    if "overview" in active:
        blocks.append(normalize(Xq_overview, norm="l2") * active["overview"])
    else:
        blocks.append(csr_matrix((1, X_overview.shape[1])))

    if "director" in active:
        blocks.append(normalize(Xq_director, norm="l2") * active["director"])
    else:
        blocks.append(csr_matrix((1, X_director.shape[1])))

    if "stars" in active:
        blocks.append(normalize(Xq_stars, norm="l2") * active["stars"])
    else:
        blocks.append(csr_matrix((1, X_stars.shape[1])))

    if "countries_of_origin" in active:
        blocks.append(normalize(Xq_origin, norm="l2") * active["countries_of_origin"])
    else:
        blocks.append(csr_matrix((1, X_origin.shape[1])))

    Xq = hstack(blocks, format="csr")
    Xq = normalize(Xq, norm="l2", copy=False)
    return Xq


## 13. Mô hình gợi ý lai (Cốt lõi) - Hàm gợi ý phim hoàn chỉnh
- Truy xuất phim ứng viên và xếp hạng lại bằng độ tương đồng nội dung kết hợp điểm đánh giá.

In [ ]:
# =========================
# 11) Hệ thống gợi ý cuối cùng: truy xuất ứng viên và xếp hạng lại (RETRIEVAL + RERANKING)
# =========================

def recommend_movies_hybrid(
    movie_title,
    release_year=None,
    top_n=10,
    candidate_pool=50,
    alpha_similarity=0.8,
    beta_weighted_rating=0.2,
    verbose=True,
):
    """
    Gợi ý phim tương tự dựa trên một bộ phim đầu vào.

    Quy trình:
    1. Xác định đúng phim người dùng muốn tìm.
    2. Dùng KNN để lấy danh sách phim có nội dung gần giống nhất. (Retrieval)
    3. Loại bỏ chính bộ phim đầu vào khỏi kết quả.
    4. Chuyển khoảng cách cosine từ KNN thành điểm tương đồng CosineSimilarity.
    5. Chuẩn hóa điểm đánh giá Weighted_Rating về khoảng 0-1.
    6. Tính FinalScore bằng cách kết hợp độ tương đồng nội dung và điểm đánh giá. (Reranking)
    7. Trả về top_n phim có FinalScore cao nhất.
    """
    
    chosen_idx, candidates = resolve_movie_query(
        movie_title=movie_title,
        release_year=release_year,
        verbose=verbose,
    )

    if chosen_idx is None:
        return None
    
    # Giai đoạn 1: Truy xuất (Retrieval) nội dung bằng KNN
    n_neighbors = min(candidate_pool + 1, len(df))
    distances, indices = nn_model.kneighbors(X_all[chosen_idx], n_neighbors=n_neighbors)

    distances = distances.ravel()
    indices = indices.ravel()

    candidate_indices = []
    cosine_scores = []

    # Lọc bỏ chính phim gốc ra khỏi danh sách gợi ý
    for dist, idx in zip(distances, indices):
        if idx == chosen_idx:
            continue
        candidate_indices.append(idx)
        cosine_scores.append(1 - dist) # Chuyển đổi Distance thành Cosine Similarity
        if len(candidate_indices) == candidate_pool:
            break

    candidate_df = df.loc[candidate_indices, SHOW_COLS].copy()
    candidate_df["CosineSimilarity"] = cosine_scores
    
    # Xử lý giá trị rating bị rỗng
    candidate_df["Weighted_Rating"] = pd.to_numeric(candidate_df["Weighted_Rating"], errors="coerce")
    candidate_df["Weighted_Rating"] = candidate_df["Weighted_Rating"].fillna(candidate_df["Weighted_Rating"].median())

    # Giai đoạn 2: Xếp hạng lại (Reranking)
    candidate_df["Weighted_Rating_Norm"] = minmax_scale_safe(candidate_df["Weighted_Rating"])
    candidate_df["FinalScore"] = (
        alpha_similarity * candidate_df["CosineSimilarity"] +
        beta_weighted_rating * candidate_df["Weighted_Rating_Norm"]
    )

    # Sắp xếp và trích xuất danh sách cuối cùng
    candidate_df = candidate_df.sort_values(
        by=["FinalScore", "CosineSimilarity", "Weighted_Rating", "Release_Year"],
        ascending=[False, False, False, False]
    ).head(top_n).reset_index(drop=True)

    return candidate_df[[
        c for c in [
            "ID", "Movie_Title", "Release_Year", "CosineSimilarity",
            "Weighted_Rating", "Weighted_Rating_Norm", "FinalScore",
            "Genre", "Director", "Stars", "Countries_of_origin"
        ] if c in candidate_df.columns
    ]]


def recommend_by_query_rerank(
    genre="",
    overview="",
    director="",
    stars="",
    countries_of_origin="",
    top_n=10,
    candidate_pool=50,
    alpha_similarity=0.8,
    beta_weighted_rating=0.2,
):

    """Hàm gợi ý dựa trên từ khóa tùy biến của người dùng (Không cần chọn tên phim gốc)."""

    Xq = build_query_vector(
        genre=genre,
        overview=overview,
        director=director,
        stars=stars,
        countries_of_origin=countries_of_origin,
    )

    n_neighbors = min(candidate_pool, len(df))
    distances, indices = nn_model.kneighbors(Xq, n_neighbors=n_neighbors)
    distances = distances.ravel()
    indices = indices.ravel()

    candidate_df = df.loc[indices, SHOW_COLS].copy()
    candidate_df["CosineSimilarity"] = 1 - distances
    candidate_df["Weighted_Rating"] = pd.to_numeric(candidate_df["Weighted_Rating"], errors="coerce")
    candidate_df["Weighted_Rating"] = candidate_df["Weighted_Rating"].fillna(candidate_df["Weighted_Rating"].median())

    candidate_df["Weighted_Rating_Norm"] = minmax_scale_safe(candidate_df["Weighted_Rating"])
    candidate_df["FinalScore"] = (
        alpha_similarity * candidate_df["CosineSimilarity"] +
        beta_weighted_rating * candidate_df["Weighted_Rating_Norm"]
    )

    candidate_df = candidate_df.sort_values(
        by=["FinalScore", "CosineSimilarity", "Weighted_Rating", "Release_Year"],
        ascending=[False, False, False, False]
    ).head(top_n).reset_index(drop=True)

    return candidate_df[[
        c for c in [
            "ID", "Movie_Title", "Release_Year", "CosineSimilarity",
            "Weighted_Rating", "Weighted_Rating_Norm", "FinalScore",
            "Genre", "Director", "Stars", "Countries_of_origin"
        ] if c in candidate_df.columns
    ]]


def recommend_movies_unified(
    movie_title=None,
    release_year=None,
    genre="",
    overview="",
    director="",
    stars="",
    countries_of_origin="",
    top_n=10,
    candidate_pool=50,
    alpha_similarity=0.8,
    beta_weighted_rating=0.2,
):

    """
    Hàm phân luồng (Unified Interface):
    - Nếu tham số 'movie_title' được truyền vào -> Chạy hàm Gợi ý dựa trên Phim.
    - Nếu không có -> Chạy hàm Gợi ý dựa trên Truy vấn tự do.
    """

    if movie_title and str(movie_title).strip():
        return recommend_movies_hybrid(
            movie_title=movie_title,
            release_year=release_year,
            top_n=top_n,
            candidate_pool=candidate_pool,
            alpha_similarity=alpha_similarity,
            beta_weighted_rating=beta_weighted_rating,
            verbose=True,
        )

    return recommend_by_query_rerank(
        genre=genre,
        overview=overview,
        director=director,
        stars=stars,
        countries_of_origin=countries_of_origin,
        top_n=top_n,
        candidate_pool=candidate_pool,
        alpha_similarity=alpha_similarity,
        beta_weighted_rating=beta_weighted_rating,
    )


## 14. Chạy thử ví dụ 

In [ ]:
# ==========================================
# VÍ DỤ 1: CHẠY THỬ MÔ HÌNH DỰA TRÊN TÊN PHIM GỐC
# ==========================================

# Ví dụ 1: title-based hybrid rerank
results_title = recommend_movies_hybrid(
    movie_title="The Batman",
    top_n=10,
    candidate_pool=50,
    alpha_similarity=0.8, # 80% quyết định bởi nội dung 
    beta_weighted_rating=0.2, # 30% quyết định bởi chất lượng phim 
    verbose=True,
)

display(results_title)

Danh sách ứng viên truy vấn:


,Movie_Title,Release_Year,Weighted_Rating
18710,The Batman,2022,7.798343


Phim được chọn làm truy vấn là index: 18710


,ID,Movie_Title,Release_Year,CosineSimilarity,Weighted_Rating,Weighted_Rating_Norm,FinalScore,Genre,Director,Stars,Countries_of_origin
0,MV33595,Hilo 3,2023,0.787513,6.289713,0.684647,0.766940,Action|Crime|Drama,Wilfred La Salle,James B. Randolph | Matthew Silva | Katina Natale,United States
1,MV9021,Nobody,2021,0.704839,7.397110,1.000000,0.763871,Action|Crime|Drama,Ilya Naishuller,Bob Odenkirk | Aleksey Serebryakov | Connie Ni...,United States|China|Japan
2,MV9326,Batman: The Long Halloween,2021,0.694517,7.250125,0.958143,0.747242,Animation|Action|Crime,Chris Palmer,Jensen Ackles | Laila Berzins | Frances Callier,United States
3,MV47608,Vettaiyan,2024,0.714398,6.885169,0.854215,0.742361,Action|Crime|Drama,T.J. Gnanavel,Rajinikanth | Amitabh Bachchan | Fahadh Faasil,India
4,MV4867,Unidentified,2021,0.731929,6.586866,0.769267,0.739396,Action|Crime|Drama,Bogdan George Apetri,Bogdan Farcas | Dragos Dumitru | Vasile Muraru,Romania
5,MV29897,Luther: The Fallen Sun,2023,0.740446,6.398298,0.715569,0.735470,Action|Crime|Drama,Jamie Payne,Idris Elba | Cynthia Erivo | Andy Serkis,United Kingdom|United States
6,MV50901,Common Creed: Trafficking,2024,0.743522,6.282283,0.682531,0.731324,Action|Crime|Drama,Javon Bates | Anthony D. Phillips Sr.,Ashlee Lawhorn | Javon Bates | David Sollberger,United States
7,MV46509,Runnin' from Wolves,2024,0.743033,6.279457,0.681726,0.730771,Action|Crime|Drama,Jay Alexander,Jay Alexander | Chris Wick | Myra Symone Popova,United States
8,MV30296,Gumraah,2023,0.725624,6.474208,0.737185,0.727937,Action|Crime|Drama,Vardhan Ketkar,Aditya Roy Kapoor | Mrunal Thakur | Ronit Roy,India
9,MV52897,Havoc,2025,0.780010,5.609820,0.491034,0.722215,Action|Crime|Drama,Gareth Evans,Tom Hardy | Jessie Mei Li | Justin Cornwell,United Kingdom|United States


In [ ]:
# ==========================================
# VÍ DỤ 2: CHẠY THỬ MÔ HÌNH DỰA TRÊN TRUY VẤN TỰ DO
# ==========================================

results_query = recommend_by_query_rerank(
    genre="Action|Sci-Fi",
    overview="A thief enters dreams to steal secrets",
    director="Christopher Nolan",
    countries_of_origin="United States of America",
    top_n=10,
    candidate_pool=50,
    alpha_similarity=0.8,
    beta_weighted_rating=0.2,
)

display(results_query)


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['united_states_of_america'] will be ignored
  warnings.warn(


,ID,Movie_Title,Release_Year,CosineSimilarity,Weighted_Rating,Weighted_Rating_Norm,FinalScore,Genre,Director,Stars,Countries_of_origin
0,MV37177,Alya,2023,0.664734,6.635000,1.000000,0.765314,Action|Sci-Fi,Jandae Percem,Jandae Percem | Edis | Hande Ataizi,United States
1,MV30993,The Tomorrow Job,2023,0.794357,5.239838,0.625041,0.743562,Action|Sci-Fi,Bruce Wemple,Grant Schumacher | Caitlin Duffy | Ariella Mas...,United States
2,MV52158,Kohinoor,2024,0.669328,6.314883,0.913967,0.742719,Action|Sci-Fi,Sourav Das,Rajatabha Dutta | Anushka Adhikari | Moubani D...,India
3,MV43278,Red vs. Blue: Restoration,2024,0.611888,6.195789,0.881959,0.692909,Action|Sci-Fi,Matt Hullum,Quentin Smith | Anna Hullum | Austin Harper,United States
4,MV61706,Azimuth,2025,0.597888,6.286122,0.906237,0.690392,Action|Sci-Fi,Thomas Teisseire,Madison Mitts | Julien Lindstrom,France
5,MV49247,SCP: The Infinite Mission,2024,0.600378,6.263238,0.900087,0.690291,Action|Sci-Fi,Sonny P. Louis,Henry Roe | Sisse Marie | Morgan Dixon,United States
6,MV4558,Kamen Rider Zero-One: Real×Time,2020,0.577354,6.461445,0.953356,0.690154,Action|Sci-Fi,Teruaki Sugihara,Fumiya Takahashi | Ryutaro Okada | Noa Tsurushima,Japan
7,MV5908,Cyborg: Deadly Machine,2020,0.595447,6.299254,0.909766,0.689742,Action|Sci-Fi,Mathieu Cailliere,Alex Tutusaus | Aurélie Aloy | Fréderic Avila,France
8,MV52425,The Winter Prophecy,2024,0.595275,6.289391,0.907115,0.688827,Action|Sci-Fi,Antione Manuel Way,Ricky Bell | Michael McNeil | Qeshone Lucas,United States
9,MV14579,Kamen Rider Zero-One Others: Kamen Rider Vulca...,2021,0.585241,6.364203,0.927221,0.687835,Action|Sci-Fi,Masaya Kakehi,Ryutaro Okada | Hiroe Igeta | Noa Tsurushima,Japan
